# Notebook 6a: JDOT (joint distribution optimal transport)

The first of our two **optimal transport** methods (Courty et al., 2017).

### Optimal transport in one picture
Imagine a pile of sand (the lab images) that has to be moved to fill a set of holes (the field images). Every grain has a
cost to travel to every hole. **Optimal transport** finds the cheapest way to move *all* the sand, and the answer is a table
**gamma**: `gamma[i, j]` says how much of lab image *i* goes to field image *j*. Images that end up linked in this plan are
treated as "the same kind of thing".

### What makes it *joint* (JDOT)
The cost of moving lab image *i* to field image *j* has **two parts**:
1. how far apart their **features** are, and
2. how badly lab image *i*'s **label** disagrees with what a classifier `f` currently predicts for field image *j*.

So a lab image of class "tomato blight" is cheap to match with a field image that *looks* similar **and** that `f` believes is
tomato blight. The plan then tells us, for every field image, which lab labels flowed into it, which gives it a **soft label**
(for example "70% blight, 30% mould"). We use those soft labels to train a better classifier `f` **on the field images**,
and repeat:

**start with f  ->  build the transport plan  ->  read off soft labels  ->  retrain f on field images  ->  repeat**

### Our setting
* Everything works on the 2048-number features (no images, no GPU).
* **Lab side:** a random subset of PlantVillage training features (3,000 of them, to keep the transport problem small).
* **Field side:** each split's ~200 **calibration** images, **without labels**. The final classifier `f` is trained on
  these ~200 images and then used for **all** PlantDoc images.
* Uses the **POT** library (`pip install POT`) for the transport step.

### Choices the paper does not fix (ours)
The weight of the feature cost against the label cost, how strongly `f` is regularised, the number of rounds, and the lab
subset size. We use sensible defaults and list them in Step 1. We do not tune them on field data.

### Plan
1. Settings
2. Load the features
3. Optimal transport on a tiny example
4. The JDOT building blocks
5. JDOT on one split, looked at closely
6. All 10 splits, then save
7. Evaluate with notebook 3b

## Step 1: Settings

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

try:
    import ot                      # POT: Python Optimal Transport
except ImportError:
    raise ImportError("Install POT first, in a terminal:  pip install POT")

In [2]:
# ---- Edit these -------------------------------------------------------------
QUICK_TEST = False                 # True = 2 splits and 2 rounds. False = the real run.

METHOD = "jdot"
SOURCE_METHOD = "source_only"
TRAIN_SEED = 42

OUT_DIR = Path("outputs")
MANIFEST = OUT_DIR / "master_manifest.csv"
SEEDS = list(range(42, 52))       # the 10 Setting B splits
DEV_SEED = 42                     # the split we look at closely in Step 5

# Our choices
SOURCE_SUBSET = 3000              # lab images used in the transport problem (random, keeps the natural class mix)
ROUNDS = 5                        # how many times to repeat: plan -> soft labels -> retrain f
ALPHA = 1.0                       # weight of the FEATURE cost relative to the LABEL cost (both are scaled to average 1)
C_TARGET = 0.1                    # regularisation of f on the ~200 field images (smaller = stronger)
C_SOURCE = 1.0                    # regularisation of the starting classifier trained on lab images
MAX_ITER = 2000                    # how long the logistic regressions may optimise
EPS = 1e-6
# -----------------------------------------------------------------------------

if QUICK_TEST:
    ROUNDS, SOURCE_SUBSET = 2, 1500
seeds_to_run = SEEDS[:2] if QUICK_TEST else SEEDS
print(f"Method: {METHOD} | splits: {seeds_to_run} | rounds: {ROUNDS} | lab subset: {SOURCE_SUBSET}")
if QUICK_TEST:
    print("*** QUICK TEST: 2 splits, 2 rounds, a smaller lab subset. Set QUICK_TEST = False for the real run. ***")

Method: jdot | splits: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51] | rounds: 5 | lab subset: 3000


## Step 2: Load the features

Same data as notebooks 4c and 5: PlantVillage training features (02b) and the PlantDoc features (02), **standardised with
the PlantVillage training statistics** (subtract the lab average, divide by the lab spread, feature by feature).
Standardising matters here because the transport cost is a **distance** between feature vectors, and we do not want a few
large-valued features to dominate it.

In [3]:
source = np.load(OUT_DIR / f"outputs_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
classes = list(source["classes"])
n_classes = len(classes)

train = np.load(OUT_DIR / f"features_pv_train_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
train_features, src_y = train["features"].astype(np.float32), train["labels"]

mean, std = train_features.mean(axis=0), train_features.std(axis=0)
standardise = lambda x: ((x - mean) / (std + EPS)).astype(np.float32)

src_z = standardise(train_features)
known_z = standardise(source["pd_known_features"].astype(np.float32))
unknown_z = standardise(source["pd_unknown_features"].astype(np.float32))
field_labels = source["pd_known_labels"]
known_paths = source["pd_known_paths"]

manifest = pd.read_csv(MANIFEST, keep_default_na=False)
split_columns = [f"B_seed{s}" for s in SEEDS]
known_rows = manifest[(manifest["dataset"] == "pd") & (manifest["role"] == "pd_known")].set_index("path")
split_info = known_rows.loc[known_paths, split_columns].reset_index(drop=True)

print("lab (source) features:", src_z.shape, "| field known:", known_z.shape, "| field unknown:", unknown_z.shape)

lab (source) features: (18601, 2048) | field known: (2018, 2048) | field unknown: (466, 2048)


## Step 3: Optimal transport on a tiny example

Three "lab" points on a line at positions 0, 1, 2 and three "field" points at 10, 11, 12. The cost of moving one point to
another is the squared distance. Each point carries one third of the mass. What is the cheapest plan?

`ot.emd(a, b, cost)` solves it (*emd* = earth mover's distance, the technical name for this problem).

In [4]:
lab_points = np.array([[0.0], [1.0], [2.0]])
field_points = np.array([[10.0], [11.0], [12.0]])

cost = ot.dist(lab_points, field_points)                  # squared distance between every pair
mass = np.ones(3) / 3                                       # every point carries 1/3 of the mass
plan = ot.emd(mass, mass, cost)

print("cost between every lab/field pair:\n", cost)
print("\ntransport plan (rows = lab points, columns = field points):\n", np.round(plan, 3))

cost between every lab/field pair:
 [[100. 121. 144.]
 [ 81. 100. 121.]
 [ 64.  81. 100.]]

transport plan (rows = lab points, columns = field points):
 [[0.333 0.    0.   ]
 [0.    0.333 0.   ]
 [0.    0.    0.333]]


Each lab point is sent to **one** field point, in order (0 to 10, 1 to 11, 2 to 12), because crossing paths would cost more.
Every row and every column of the plan adds up to 1/3. With *many* points, a lab point's mass is usually **split** among
several field points, and that is what gives the soft labels below.

## Step 4: The JDOT building blocks

**1. The transport plan** for the current classifier `f`. The cost has the two parts described above, each scaled to have
an average of 1 so that `ALPHA` is a fair balance:
* feature cost: squared distance between the two feature vectors;
* label cost: `-log` of the probability `f` gives to lab image *i*'s label on field image *j*. If `f` is sure that field
  image *j* is a different class, the cost is high.

In [5]:
def transport_plan(lab_z, lab_y, field_z, f):
    feature_cost = ot.dist(lab_z, field_z) / lab_z.shape[1]
    feature_cost = feature_cost / feature_cost.mean()

    probs = np.clip(f.predict_proba(field_z), 1e-6, 1.0)             # what f believes about each field image
    label_cost = -np.log(probs[:, lab_y].T)                          # (lab images x field images)
    label_cost = label_cost / label_cost.mean()

    cost = ALPHA * feature_cost + label_cost
    lab_mass = np.ones(len(lab_z)) / len(lab_z)
    field_mass = np.ones(len(field_z)) / len(field_z)
    return ot.emd(lab_mass, field_mass, cost), cost

**2. Soft labels.** Look down column *j* of the plan: it says how much of each lab image flowed into field image *j*. Adding up
the **labels** of those lab images, weighted by the flow, gives field image *j* a distribution over classes that sums to 1.

In [6]:
def soft_labels(plan, lab_y):
    one_hot = np.eye(n_classes)[lab_y]                               # (lab images x classes)
    return (plan.T @ one_hot) * plan.shape[1]                        # each field image's labels now sum to 1

**3. Retrain `f` on the field images** using those soft labels. A logistic regression normally takes one label per image. To
use a *distribution*, we make 21 copies of every field image, one per class, and weight each copy by the soft label for
that class (a standard trick). A strong penalty (`C_TARGET`) keeps `f` from memorising ~200 images in 2048 dimensions.

In [7]:
def fit_field_classifier(field_z, soft):
    n_field = len(field_z)
    X = np.repeat(field_z, n_classes, axis=0)                        # 21 copies of every field image
    y = np.tile(np.arange(n_classes), n_field)                       # ... one per class
    weight = np.maximum(soft.reshape(-1), 1e-8)                      # ... weighted by the soft label (tiny floor keeps every class present)
    return LogisticRegression(C=C_TARGET, max_iter=MAX_ITER).fit(X, y, sample_weight=weight)

**4. The starting classifier** `f0`: an ordinary classifier trained on all the lab images. It gives the first round
something to compare labels against. (This takes a minute or two, once.)

In [8]:
start = time.time()
f0 = LogisticRegression(C=C_SOURCE, max_iter=MAX_ITER).fit(src_z, src_y)
print(f"Starting classifier trained in {time.time() - start:.0f} s")
assert list(f0.classes_) == list(range(n_classes)), "Every class must appear in the lab training data"

Starting classifier trained in 2 s


In [9]:
def run_jdot(field_z, seed, verbose=False):
    rng = np.random.default_rng(seed)
    subset = rng.choice(len(src_z), size=min(SOURCE_SUBSET, len(src_z)), replace=False)
    lab_z, lab_y = src_z[subset], src_y[subset]

    f = f0
    for round_number in range(1, ROUNDS + 1):
        plan, cost = transport_plan(lab_z, lab_y, field_z, f)
        soft = soft_labels(plan, lab_y)
        f = fit_field_classifier(field_z, soft)
        if verbose:
            agree = float((f.predict(field_z) == soft.argmax(axis=1)).mean())
            print(f"round {round_number}: transport cost {float((plan * cost).sum()):.3f} | "
                  f"f agrees with the transported labels on {agree * 100:.0f}% of the field images")
    return f

## Step 5: JDOT on one split, looked at closely

In [10]:
seed = DEV_SEED
is_cal = (split_info[f"B_seed{seed}"] == "cal").to_numpy()
is_test = ~is_cal
field_calibration = known_z[is_cal]                                  # this split's ~200 field images, NO labels
print(f"Split {seed}: {len(field_calibration)} unlabelled field images, {SOURCE_SUBSET} lab images in the transport problem")

start = time.time()
f_jdot = run_jdot(field_calibration, seed, verbose=True)
print(f"JDOT finished in {time.time() - start:.0f} s")

Split 42: 202 unlabelled field images, 3000 lab images in the transport problem
round 1: transport cost 0.962 | f agrees with the transported labels on 100% of the field images
round 2: transport cost 0.577 | f agrees with the transported labels on 100% of the field images
round 3: transport cost 0.572 | f agrees with the transported labels on 100% of the field images
round 4: transport cost 0.572 | f agrees with the transported labels on 100% of the field images
round 5: transport cost 0.571 | f agrees with the transported labels on 100% of the field images
JDOT finished in 34 s


In [11]:
def accuracy(logits, mask):
    return float((logits[mask].argmax(axis=1) == field_labels[mask]).mean())

start_logits = f0.decision_function(known_z)                         # the lab-trained classifier f0, no adaptation
final_logits = f_jdot.decision_function(known_z)                     # the classifier JDOT ended with

print(f"Accuracy on the {is_test.sum()} test images (labels used only for scoring):")
print(f"  starting classifier f0 (lab only): {accuracy(start_logits, is_test):.4f}")
print(f"  after JDOT:                        {accuracy(final_logits, is_test):.4f}")

Accuracy on the 1816 test images (labels used only for scoring):
  starting classifier f0 (lab only): 0.2214
  after JDOT:                        0.2583


**Questions to think about:**
* `f` is trained on only ~200 field images. How could that hurt, and what stops it from simply memorising them?
* Optimal transport forces **all** lab mass to be moved, and every lab image carries the same weight. What happens to a
  disease that is common in the lab data but rare (or absent) in the field? Think about where its mass has to go.
* Compare with `linear_head` in notebook 3b (a classifier trained on lab features only). Is JDOT's change in accuracy
  larger than what re-training a classifier alone gives?

## Step 6: All splits, then save

For each split, we run JDOT with that split's calibration images (no labels) and keep the final classifier's scores for all
known and unknown PlantDoc images.

In [12]:
save_dict = {"classes": np.array(classes)}
for key in ["epochs_run", "debug_mode"]:
    if key in source.files:
        save_dict[key] = source[key]
for g in ["pv_val", "pv_test"]:
    save_dict[f"{g}_logits"] = source[f"{g}_logits"]
for g in ["pv_val", "pv_test", "pd_known", "pd_unknown"]:
    save_dict[f"{g}_labels"] = source[f"{g}_labels"]
    save_dict[f"{g}_paths"] = source[f"{g}_paths"]

for seed in seeds_to_run:
    start = time.time()
    is_cal = (split_info[f"B_seed{seed}"] == "cal").to_numpy()
    f = run_jdot(known_z[is_cal], seed)

    save_dict[f"pd_known_logits_B_seed{seed}"] = f.decision_function(known_z).astype(np.float32)
    save_dict[f"pd_unknown_logits_B_seed{seed}"] = f.decision_function(unknown_z).astype(np.float32)
    print(f"split {seed}: {time.time() - start:.0f} s | test accuracy "
          f"{accuracy(save_dict[f'pd_known_logits_B_seed{seed}'], ~is_cal):.4f}")

suffix = "_QUICK" if QUICK_TEST else ""
output_file = OUT_DIR / f"outputs_{METHOD}_seed{TRAIN_SEED}{suffix}.npz"
np.savez_compressed(output_file, **save_dict)
print("Saved", output_file)

split 42: 32 s | test accuracy 0.2583
split 43: 39 s | test accuracy 0.2599
split 44: 34 s | test accuracy 0.2682
split 45: 46 s | test accuracy 0.2605
split 46: 43 s | test accuracy 0.2645
split 47: 46 s | test accuracy 0.2467
split 48: 35 s | test accuracy 0.2771
split 49: 35 s | test accuracy 0.2561
split 50: 45 s | test accuracy 0.2704
split 51: 36 s | test accuracy 0.2665
Saved outputs\outputs_jdot_seed42.npz


## Step 7: Evaluate with notebook 3b

Run once with `QUICK_TEST = True`, then set it to `False`, restart the kernel and run again. The real file is
`outputs_jdot_seed0.npz`. Add `"jdot"` to `METHODS` in notebook 3b.

**How to read JDOT's rows.** JDOT's final classifier is a **new logistic regression**, trained on field images, so the fair
comparisons are:
* `linear_head`: a logistic regression trained on lab features only (no field images at all);
* `source_only`: the fine-tuned network.

The paper's baselines did not include any optimal transport method, so there are no reference values. This is part of our
contribution.

### Next
**Notebook 6b: DeepJDOT**, which does the same thing *during* training of the network, one mini-batch at a time.